In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core.base_options import BaseOptions
from src.feature_extraction import FeatureExtractor

f=FeatureExtractor()

YAW_THRESHOLD   = 20.0   # left / right
PITCH_THRESHOLD = 15.0   # up / down
ROLL_THRESHOLD  = 15.0   # tilt (optional)

# -----------------------------
# Webcam
# -----------------------------

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame=frame.copy() # avoid modifying the original frame
    model_image=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = frame.shape[:2]
    
    features=f._extract_head_pose(model_image,w,h)

    if features:
        rvec=features["rvec"]
        tvec=features["tvec"]
        cam_matrix=features["camera_matrix"]
        dist_coeffs=features["dist_coeffs"]
        nose_2d=features["nose_2d"]
        pitch=features["head_pitch"]
        yaw=features["head_yaw"]
        roll=features["head_roll"]
        head_pose=features["head_pose"]
        
        cv2.putText(frame, f"Pitch: {pitch:.1f}", (20, 40),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, f"Yaw: {yaw:.1f}", (20, 70),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, f"Roll: {roll:.1f}", (20, 100),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, f"Head Pose: {head_pose}", (20, 130),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        cv2.putText(frame, f"yaw,pitch,roll in radians: {yaw:.2f}, {pitch :.2f}, {roll:.2f}", (20, 160),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
        
        axis = np.float64([
            [50, 0, 0],
            [0, 50, 0],
            [0, 0, 50] ]
            )
        
        imgpts, _ = cv2.projectPoints( axis, rvec, tvec, cam_matrix, dist_coeffs )
        nose = tuple(map(int, nose_2d))
        print("frame id:", id(frame), "nose:", nose_2d)

        cv2.line(frame, nose, tuple(imgpts[0].ravel().astype(int)), (0,0,255), 3)
        cv2.line(frame, nose, tuple(imgpts[1].ravel().astype(int)), (0,255,0), 3)
        cv2.line(frame, nose, tuple(imgpts[2].ravel().astype(int)), (255,0,0), 3)
        
    cv2.imshow("Head Pose Estimation", frame)
    if cv2.waitKey(1) & 0xFF == 27: # ESC key to exit
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")  # This downloads the YOLO26n weights for you

In [ ]:
import Dict

def _extract_gaze_features(self, rgb_image: np.ndarray, width: int, height: int) -> Dict:

    """
    Extract only one gaze feature.

    gaze_direction:
        0 = on screen OR on script
        1 = away from screen OR unknown / not detected

    Notes:
        - "screen" means candidate is looking approximately toward the screen region.
        - "script" means candidate is looking downward, probably at script/book/table.
        - both screen and script are treated as 0 as requested.
        - away / unknown / no face / solvePnP failure are treated as 1.
    """

    # Only one output feature
    features = {
        "gaze_direction": 1
    }

    # --------------------------------------------------
    # Run MediaPipe Face Landmarker / Face Mesh
    # --------------------------------------------------
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_image
    )

    result = self.face_mesh.detect(mp_image)

    # If no face is detected, mark as unknown/away
    if not result.face_landmarks:
        features["gaze_direction"] = 1
        return features

    face_landmarks = result.face_landmarks[0]

    # Iris landmarks require at least 478 landmarks
    if len(face_landmarks) < 478:
        features["gaze_direction"] = 1
        return features

    # --------------------------------------------------
    # Helper functions
    # --------------------------------------------------
    def landmark_xy(index: int) -> tuple:
        return (
            float(face_landmarks[index].x * width),
            float(face_landmarks[index].y * height)
        )

    def iris_center(indices: list) -> tuple:
        points = np.array([landmark_xy(i) for i in indices], dtype=np.float64)
        center_x, center_y = points.mean(axis=0)
        return float(center_x), float(center_y)

    # --------------------------------------------------
    # Iris / pupil center
    # --------------------------------------------------
    LEFT_IRIS = [468, 469, 470, 471, 472]
    RIGHT_IRIS = [473, 474, 475, 476, 477]

    left_iris_x, left_iris_y = iris_center(LEFT_IRIS)
    right_iris_x, right_iris_y = iris_center(RIGHT_IRIS)

    gaze_point_x = (left_iris_x + right_iris_x) / 2.0
    gaze_point_y = (left_iris_y + right_iris_y) / 2.0

    gx_norm = gaze_point_x / float(width)
    gy_norm = gaze_point_y / float(height)

    # --------------------------------------------------
    # Head pose estimation using solvePnP
    # --------------------------------------------------
    model_points = np.array([
        (0.0, 0.0, 0.0),          # Nose tip
        (0.0, -63.6, -12.5),     # Chin
        (-43.3, 32.7, -26.0),    # Left eye corner
        (43.3, 32.7, -26.0),     # Right eye corner
        (-28.9, -28.9, -24.1),   # Left mouth corner
        (28.9, -28.9, -24.1),    # Right mouth corner
    ], dtype=np.float64)

    focal_length = float(width)
    image_center = (width / 2.0, height / 2.0)

    camera_matrix = np.array([
        [focal_length, 0.0, image_center[0]],
        [0.0, focal_length, image_center[1]],
        [0.0, 0.0, 1.0],
    ], dtype=np.float64)

    dist_coeffs = np.zeros((4, 1), dtype=np.float64)

    success, rvec, tvec = cv2.solvePnP(
        model_points,
        image_points,
        camera_matrix,
        dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE
    )

    if not success:
        features["gaze_direction"] = 1
        return features

    # --------------------------------------------------
    # Project head direction point
    # --------------------------------------------------
    gaze_3d = np.array([[0.0, 0.0, 1000.0]], dtype=np.float64)

    gaze_2d, _ = cv2.projectPoints(
        gaze_3d,
        rvec,
        tvec,
        camera_matrix,
        dist_coeffs
    )

    gaze_2d_x, gaze_2d_y = gaze_2d[0][0]
    dx = float(gaze_2d_x - image_center[0])
    dy = float(gaze_2d_y - image_center[1])

    dx_norm = dx / float(width)
    dy_norm = dy / float(height)

    # --------------------------------------------------
    # Boundary rules
    # --------------------------------------------------

    # Screen region: middle area of the image
    SCREEN_X_MIN = 0.20
    SCREEN_X_MAX = 0.80
    SCREEN_Y_MIN = 0.15
    SCREEN_Y_MAX = 0.62

    # Script/table region: lower part of image
    SCRIPT_Y_THRESHOLD = 0.62

    # Head direction should be roughly centered
    HEAD_CENTER_X_THRESHOLD = 0.18
    HEAD_CENTER_Y_THRESHOLD = 0.18

    gaze_inside_screen = (
        SCREEN_X_MIN <= gx_norm <= SCREEN_X_MAX and
        SCREEN_Y_MIN <= gy_norm <= SCREEN_Y_MAX
    )

    head_centered = (
        abs(dx_norm) <= HEAD_CENTER_X_THRESHOLD and
        abs(dy_norm) <= HEAD_CENTER_Y_THRESHOLD
    )

    looking_at_screen = gaze_inside_screen and head_centered

    looking_at_script = (
        gy_norm >= SCRIPT_Y_THRESHOLD or
        dy_norm >= HEAD_CENTER_Y_THRESHOLD
    )

    # --------------------------------------------------
    # Final one-feature decision
    #
    # 0 = on screen OR on script
    # 1 = away OR unknown
    # --------------------------------------------------
    if looking_at_screen or looking_at_script:
        features["gaze_direction"] = 0
    else:
        features["gaze_direction"] = 1

    return features

In [3]:
import cv2
import numpy as np
from src.feature_extraction import FeatureExtractor

# Initialize extractor
extractor = FeatureExtractor()

def visualize_gaze(frame_bgr):
    h, w = frame_bgr.shape[:2]

    # MediaPipe expects RGB
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    features = extractor._extract_gaze_features(frame_rgb, w, h)

    # ---------- Draw pupils ----------
    if features["pupil_left_x"] > 0:
        cv2.circle(frame_bgr, (int(features["pupil_left_x"]), int(features["pupil_left_y"])), 3, (0, 255, 0), -1 )

    if features["pupil_right_x"] > 0:
        cv2.circle(
            frame_bgr,
            (int(features["pupil_right_x"]), int(features["pupil_right_y"])),
            3,
            (0, 255, 0),
            -1
        )

    # ---------- Draw gaze point ----------
    gaze_x = int(features["gazePoint_x"])
    gaze_y = int(features["gazePoint_y"])

    if gaze_x > 0:
        cv2.circle(frame_bgr, (gaze_x, gaze_y), 5, (255, 0, 0), -1)

        # Draw arrow from face center → gaze point
        face_center = (w // 2, h // 2)
        cv2.arrowedLine(
            frame_bgr,
            face_center,
            (gaze_x, gaze_y),
            (0, 0, 255),
            2,
            tipLength=0.2
        )

    # ---------- Text info ----------
    cv2.putText(
        frame_bgr,
        f"Gaze: {features['gaze_direction']}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # cv2.putText(
    #     frame_bgr,
    #     f"Gaze_directions: {features['dx']:.2f}, {features['dy']:.2f}",
    #     (40, 30),
    #     cv2.FONT_HERSHEY_SIMPLEX,
    #     0.7,
    #     (255, 255, 255),
    #     2
    # )
    
    cv2.putText(
        frame_bgr,
        f"On Script: {features['gaze_on_script']}",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )
    return frame_bgr

In [4]:
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = visualize_gaze(frame)
    cv2.imshow("Gaze Visualization", frame)
    
    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break
    
cap.release()
cv2.destroyAllWindows()

In [ ]:
def _extract_gaze_features(self, rgb_image: np.ndarray, width: int, height: int) -> Dict:
        """Extract gaze-related features"""
        features = {'gaze_direction': 1} #,'dx':0,'dy':0}
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb_image)
        result = self.face_mesh.detect(mp_image)

        if not result.face_landmarks:
            # features['gaze_on_script'] = 0
            features['gaze_direction'] = 1
            return features
        
        face_landmarks = result.face_landmarks[0]
        # Iris Landmarks:

        LEFT_IRIS = [468, 469, 470, 471, 472]
        RIGHT_IRIS = [473, 474, 475, 476, 477]
        
        if len(face_landmarks) < 478:
            features["gaze_direction"] = 1
            return features
        def landmark_xy(index: int) -> tuple:
            return (
            float(face_landmarks[index].x * width),
            float(face_landmarks[index].y * height)
        )

        def iris_center(indices: list) -> tuple:
            points = np.array([landmark_xy(i) for i in indices], dtype=np.float64)
            center_x, center_y = points.mean(axis=0)
            return float(center_x), float(center_y)
        
        # def iris_center(ids):
        #     xs = [face_landmarks[i].x * width for i in ids]
        #     ys = [face_landmarks[i].y * height for i in ids]
        #     return float(np.mean(xs)), float(np.mean(ys))

        left_iris_x, left_iris_y = iris_center(LEFT_IRIS)
        right_iris_x, right_iris_y = iris_center(RIGHT_IRIS)

        gaze_point_x = (left_iris_x + right_iris_x) / 2.0
        gaze_point_y = (left_iris_y + right_iris_y) / 2.0
        
        gx_norm = gaze_point_x / float(width)
        gy_norm = gaze_point_y / float(height)
        
        features['gazePoint_x'] = int(gaze_point_x)
        features['gazePoint_y'] = int(gaze_point_y)
        # are they useful?
        # Head Pose Estimation using solve

        model_points = np.array([
        (0.0, 0.0, 0.0),        # Nose tip
        (0.0, -63.6, -12.5),   # Chin
        (-43.3, 32.7, -26.0),  # Left eye corner
        (43.3, 32.7, -26.0),   # Right eye corner
        (-28.9, -28.9, -24.1), # Left mouth
        (28.9, -28.9, -24.1)   # Right mouth
        ], dtype=np.float64)

        image_points = np.array([
        landmark_xy(4),      # Nose tip
        landmark_xy(152),    # Chin
        landmark_xy(33),     # Left eye corner
        landmark_xy(263),    # Right eye corner
        landmark_xy(61),     # Left mouth corner
        landmark_xy(291),    # Right mouth corner
        ], dtype=np.float64)

        focal_length = float(width)
        center=(width/2.0, height/2.0)
        camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]], dtype=np.float64)
        dist_coeffs = np.zeros((4,1))  # assume no lens distortion

        success, rvec, tvec = cv2.solvePnP(
        model_points,
        image_points,
        camera_matrix,
        dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE)
        
        if not success:
            features['gaze_direction'] = 1 #Away from the script :--->
            return features
        
        gaze_3d= np.array([[0,0,1000.0]], dtype=np.float64)
        gaze_2d, _ = cv2.projectPoints(gaze_3d, rvec, tvec, camera_matrix, dist_coeffs)
        gaze_2d_x,gaze_2d_y = gaze_2d[0][0]
        # gaze direction classification

        dx=float(gaze_2d_x - center[0])
        dy=float(gaze_2d_y - center[1])
        
        dx_norm = dx / float(width)
        dy_norm = dy / float(height)

        # --------------------------------------------------
        # Boundary rules
        # --------------------------------------------------
        # Screen region: middle area of the image
        SCREEN_X_MIN = 0.20
        SCREEN_X_MAX = 0.80
        SCREEN_Y_MIN = 0.15
        SCREEN_Y_MAX = 0.62

        # Script/table region: lower part of image
        SCRIPT_Y_THRESHOLD = 0.62

        # Head direction should be roughly centered
        HEAD_CENTER_X_THRESHOLD = 0.18
        HEAD_CENTER_Y_THRESHOLD = 0.18

        gaze_inside_screen = (
            SCREEN_X_MIN <= gx_norm <= SCREEN_X_MAX and
            SCREEN_Y_MIN <= gy_norm <= SCREEN_Y_MAX
        )

        head_centered = (
            abs(dx_norm) <= HEAD_CENTER_X_THRESHOLD and
            abs(dy_norm) <= HEAD_CENTER_Y_THRESHOLD
        )

        looking_at_screen = gaze_inside_screen and head_centered

        looking_at_script = (
            gy_norm >= SCRIPT_Y_THRESHOLD or
            dy_norm >= HEAD_CENTER_Y_THRESHOLD
        )
    # --------------------------------------------------
    # Final one-feature decision
    # 0 = on screen OR on script
    # 1 = away OR unknown
    # --------------------------------------------------
        if looking_at_screen or looking_at_script:
            features["gaze_direction"] = 0
        else:
            features["gaze_direction"] = 1
        return features
    

In [ ]:
import pandas as pd

# Load the dataset
file_path = r"C:\Users\Tharun\Desktop\FRAUD DETECTION SYSTEM FOR THE ONLINE PROCTORED EXAMS\New_setup\new_data\final_window_dataset.csv"
df = pd.read_csv(file_path)

# Update the label value based on the condition
df.loc[df['phone_present_ratio'] >= 0.35, 'label'] = 1
df.loc[df['multiple_faces_ratio'] >= 0.5, 'label'] = 1

# Save the updated dataset back to the file (optional)
df.to_csv(file_path, index=False)

In [ ]:
def _extract_gaze_features(self, rgb_image: np.ndarray, width: int, height: int) -> Dict:
        """Extract gaze-related features"""
        features = {'gaze_direction': 0} #,'dx':0,'dy':0}
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb_image)
        result = self.face_mesh.detect(mp_image)

        if not result.face_landmarks:
            # features['gaze_on_script'] = 0
            features['gaze_direction'] = 1
            return features
        
        face_landmarks = result.face_landmarks[0]
        # Iris Landmarks:

        LEFT_IRIS = [468, 469, 470, 471, 472]
        RIGHT_IRIS = [473, 474, 475, 476, 477]
        
        def iris_center(ids):
            xs = [face_landmarks[i].x * width for i in ids]
            ys = [face_landmarks[i].y * height for i in ids]
            return float(np.mean(xs)), float(np.mean(ys))

        left_iris_x, left_iris_y = iris_center(LEFT_IRIS)
        right_iris_x, right_iris_y = iris_center(RIGHT_IRIS)
        
        # 1. Pupil positions
        # features['pupil_left_x'] = left_iris_x
        # features['pupil_left_y'] = left_iris_y
        # features['pupil_right_x'] = right_iris_x
        # features['pupil_right_y'] = right_iris_y
        # 2. Eye center (average pupil)

        gaze_point_x = (left_iris_x + right_iris_x) / 2
        gaze_point_y = (left_iris_y + right_iris_y) / 2

        features['gazePoint_x'] = int(gaze_point_x)
        features['gazePoint_y'] = int(gaze_point_y)
        # are they useful?
        # Head Pose Estimation using solve

        model_points = np.array([
        (0.0, 0.0, 0.0),        # Nose tip
        (0.0, -63.6, -12.5),   # Chin
        (-43.3, 32.7, -26.0),  # Left eye corner
        (43.3, 32.7, -26.0),   # Right eye corner
        (-28.9, -28.9, -24.1), # Left mouth
        (28.9, -28.9, -24.1)   # Right mouth
        ], dtype=np.float64)

        image_points = np.array([
        (face_landmarks[4].x * width,   face_landmarks[4].y * height),    # Nose tip
        (face_landmarks[152].x * width, face_landmarks[152].y * height),  # Chin
        (face_landmarks[33].x * width,  face_landmarks[33].y * height),   # Left eye corner
        (face_landmarks[263].x * width, face_landmarks[263].y * height),  # Right eye corner
        (face_landmarks[61].x * width,  face_landmarks[61].y * height),   # Left mouth
        (face_landmarks[291].x * width, face_landmarks[291].y * height)   # Right mouth
        ], dtype=np.float64)
        
        focal_length = width
        center=(width/2, height/2)
        camera_matrix = np.array([
        [focal_length, 0, center[0]],
        [0, focal_length, center[1]],
        [0, 0, 1]], dtype=np.float64)
        dist_coeffs = np.zeros((4,1))  # assume no lens distortion

        success, rvec, tvec = cv2.solvePnP(
        model_points,
        image_points,
        camera_matrix,
        dist_coeffs,
        flags=cv2.SOLVEPNP_ITERATIVE)
        
        if not success:
            features['gaze_direction'] = 1 #Away from the script :--->
            return features
        
        gaze_3d= np.array([[0,0,1000.0]])
        gaze_2d, _ = cv2.projectPoints(gaze_3d, rvec, tvec, camera_matrix, dist_coeffs)

        gaze_2d_x,gaze_2d_y = gaze_2d[0][0]
        # gaze direction classification

        dx=gaze_2d_x - center[0]
        dy=gaze_2d_y - center[1]

        CENTER_TH_X = 0.08 * width    # ~8% of width
        CENTER_TH_Y = 0.08 * height   # ~8% of height
        DIR_TH_X = 0.12 * width       # directional threshold
        DIR_TH_Y = 0.12 * height

        if abs(dx) < CENTER_TH_X and abs(dy) < CENTER_TH_Y:
            gaze_direction = 'center'
        elif abs(dx) < DIR_TH_X or abs(dy) < DIR_TH_Y:
            # Ambiguous direction (not centered, but not strongly diagonal)
            gaze_direction = 'None'
        elif dx >= 0 and dy >= 0:
            gaze_direction = 'top_right'
        elif dx < 0 and dy >= 0:
            gaze_direction = 'top_left'
        elif dx >= 0 and dy < 0:
            gaze_direction = 'bottom_right'
        
        else:
            gaze_direction = 'bottom_left'
    
        if gaze_direction in ['center','bottom_right','bottom_left','None']:
            features['gaze_direction'] = 0
        else:
            features['gaze_direction'] = 1 
        return features
